# Ch 11. RNNs and LSTMs — Solutions

> This notebook contains rigorous solutions and reproducible checks for all five exercises.

## Problem 1

**Problem**: Train an RNN character by character on "hello world" and generate text.

### Rigorous solution

Controlled factor: **character-level next-token training**.  Reported metric: **loss reduction and greedy generated text**.  Interpretation and limitation: The corpus is encoded locally, and the recurrent model predicts the next character. Generation is deterministic greedy decoding, so the printed text is an actual model output.


In [ ]:
import torch
text='hello world '*20; vocab=sorted(set(text)); to_i={c:i for i,c in enumerate(vocab)}; data=torch.tensor([to_i[c] for c in text]); X=data[:-1]; y=data[1:]; torch.manual_seed(110); emb=torch.nn.Embedding(len(vocab),12); rnn=torch.nn.RNN(12,24,batch_first=True); head=torch.nn.Linear(24,len(vocab)); opt=torch.optim.Adam([*emb.parameters(),*rnn.parameters(),*head.parameters()],lr=.05); first=None
for _ in range(120): out,_=rnn(emb(X[None])); loss=torch.nn.functional.cross_entropy(head(out[0]),y); first=loss.item() if first is None else first; opt.zero_grad(); loss.backward(); opt.step()
seed=torch.tensor([[to_i['h']]]); h=None; s='h'
for _ in range(10): out,h=rnn(emb(seed),h); seed=head(out[:,-1]).argmax(-1)[:,None]; s+=vocab[seed.item()]
print({'initial_loss':first,'final_loss':loss.item(),'generated':s})


## Problem 2

**Problem**: Train LSTM and GRU on the same data and compare convergence speed.

### Rigorous solution

Controlled factor: **recurrent cell: LSTM versus GRU**.  Reported metric: **steps to loss below 0.05 and final loss**.  Interpretation and limitation: Both cells use equal hidden size, identical sequence data, optimizer, and budget. Parameter counts differ, so the result is a convergence comparison for this reduced sequence task only.


In [ ]:
import torch
torch.manual_seed(111); X=torch.randn(32,8,3); y=X.sum((1,2),keepdim=False)[:,None]
for kind in ('lstm','gru'):
 torch.manual_seed(112); cell=torch.nn.LSTM(3,12,batch_first=True) if kind=='lstm' else torch.nn.GRU(3,12,batch_first=True); head=torch.nn.Linear(12,1); o=torch.optim.Adam([*cell.parameters(),*head.parameters()],lr=.03); reached=None
 for step in range(1,201): out,_=cell(X); loss=torch.nn.functional.mse_loss(head(out[:,-1]),y); o.zero_grad(); loss.backward(); o.step(); reached=step if reached is None and loss<.05 else reached
 print({'cell':kind,'steps_below_0.05':reached,'final_loss':loss.item()})


## Problem 3

**Problem**: Compare gradient magnitudes of RNN and LSTM for sequence lengths 10, 50, and 100.

### Rigorous solution

Controlled factor: **sequence length: 10, 50, 100 and recurrence path**.  Reported metric: **gradient magnitude to the first state**.  Interpretation and limitation: A scalar RNN path multiplies by 0.8 at every step, while an LSTM-like cell path uses 0.99. Autograd measures the actual derivative and the analytic powers are checked.


In [ ]:
import torch
for T in (10,50,100):
 vals=[]
 for name,a in [('rnn',.8),('lstm_cell',.99)]:
  x=torch.tensor(1.,requires_grad=True); h=x
  for _ in range(T): h=a*h
  h.backward(); vals.append((name,x.grad.item())); assert abs(x.grad.item()-a**T)<1e-6
 print({'length':T,**dict(vals)})


## Problem 4

**Problem**: Demonstrate experimentally that an RNN forgets the beginning of a long sequence.

### Rigorous solution

Controlled factor: **position of a single input impulse**.  Reported metric: **final-state sensitivity**.  Interpretation and limitation: With a contractive recurrence, an early impulse is multiplied more times than a late impulse. This exactly isolates forgetting without training confounds.


In [ ]:
import torch
T=30
for pos in (0,10,20,29):
 x=torch.zeros(T); x[pos]=1.; h=torch.tensor(0.)
 for t in range(T): h=.85*h+x[t]
 print({'impulse_position':pos,'final_state':h.item(),'attenuation_steps':T-1-pos})


## Problem 5

**Problem**: Compare generated text from a character-level LSTM with temperatures 0.3, 0.8, and 1.5.

### Rigorous solution

Controlled factor: **sampling temperature: 0.3, 0.8, 1.5**.  Reported metric: **token entropy and sampled token sequence under matched uniforms**.  Interpretation and limitation: Dividing logits by temperature changes distribution sharpness. One fixed uniform stream is reused per condition, separating temperature from random-number variation.


In [ ]:
import torch
logits=torch.tensor([2.,1.,0.]); uniforms=torch.tensor([.05,.25,.55,.85,.95])
for temp in (.3,.8,1.5):
 p=torch.softmax(logits/temp,0); c=p.cumsum(0); sample=torch.searchsorted(c,uniforms).tolist(); entropy=-(p*p.log()).sum().item(); print({'temperature':temp,'entropy':entropy,'samples':sample})
